# The say–do gap in mental wellness apps

**Positioning and messaging analysis for Calm, benchmarked against Headspace and Intellect**

This notebook mines 200,972 Google Play reviews of 73 mental health apps to find the gap
between what these brands advertise and what their users actually complain about, then turns
that gap into a positioning recommendation for one app.

**Data.** MHARD (Mental Health App Reviews Dataset), 200,972 reviews of 73 apps with 100k+
downloads, posted 29 March 2011 to 11 July 2023, collected via SerpAPI. MIT licensed.
Source: https://github.com/Sensify-Lab/MHARD — paper: https://doi.org/10.1609/icwsm.v19i1.35916

**Method.** Unsupervised NMF topic discovery on TF-IDF vectors to find what themes exist,
then an auditable regex theme dictionary derived from those topics to measure prevalence.
The dictionary approach is deliberate: every number below can be traced to a rule you can
read, challenge and change. Black-box clustering cannot be audited by a reader.

**Caveat carried throughout.** Reviews end July 2023. Marketing copy was collected in 2026.
The Wayback Machine was unreachable during collection, so era-matched 2023 homepage copy
could not be verified. Claims marked CURRENT are 2026; where 2022–23 wording is quoted it
comes from dated primary sources, not archived snapshots.

## 1. Load and profile

In [ ]:
import pandas as pd, numpy as np, re, urllib.request
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
import matplotlib.pyplot as plt

URL = 'https://raw.githubusercontent.com/Sensify-Lab/MHARD/main/MHARD_dataset.csv'
try:
    df = pd.read_csv('MHARD_dataset.csv', low_memory=False)
except FileNotFoundError:
    urllib.request.urlretrieve(URL, 'MHARD_dataset.csv')
    df = pd.read_csv('MHARD_dataset.csv', low_memory=False)

df['dt'] = pd.to_datetime(df['date'], format='%B %d, %Y', errors='coerce')
df['year'] = df.dt.dt.year
df = df[df.review.notna()].copy()
df['review'] = df.review.astype(str).str.strip()
print(df.shape, df.dt.min().date(), '->', df.dt.max().date())
df.app_name.value_counts().head(10)

In [ ]:
FOCUS = ['calm', 'headspace', 'intellect']
f = df[df.app_name.isin(FOCUS)]
prof = f.groupby('app_name').agg(
    n=('UID','size'), mean_rating=('rating','mean'),
    pct_1_2=('rating', lambda s: (s<=2).mean()*100),
    pct_5=('rating', lambda s: (s==5).mean()*100),
    median_words=('review', lambda s: s.str.split().str.len().median()),
    first=('dt','min'), last=('dt','max')).round(2)
prof

Calm and Headspace are sharply bimodal — roughly 30% one-star against 40%+ five-star.
Intellect sits at 4.70 with only 3.2% negative, which turns out to be a distribution artefact
rather than a quality difference (see section 6). Note the windows differ: Calm and Headspace
reviews begin September 2018, Intellect April 2020.

## 2. Unsupervised theme discovery (NMF)

Run first, so themes come out of the corpus rather than out of the analyst's assumptions.

In [ ]:
BRAND = ['headspace','calm','calmness','intellect','app','apps','application','meditation',
 'meditate','meditating','meditations','use','using','used','really','just','like','good',
 'great','love','nice','best','thing','things','way','ive','im','dont','doesnt','didnt',
 'cant','wont','lot','make','makes','time','times','day','days','know','want','need','ll',
 've','got','going','say','said','little','bit']
stop = list(ENGLISH_STOP_WORDS.union(BRAND))
vec = TfidfVectorizer(strip_accents='unicode', stop_words=stop, ngram_range=(1,2),
                      min_df=25, max_df=0.35, max_features=40000, token_pattern=r'[a-z][a-z]+')
X = vec.fit_transform(f.review)
terms = np.array(vec.get_feature_names_out())
nmf = NMF(n_components=18, init='nndsvd', random_state=0, max_iter=500)
W = nmf.fit_transform(X); H = nmf.components_
for i in range(18):
    print(f'T{i:02d} [{(W.argmax(1)==i).mean()*100:4.1f}%] ' +
          ', '.join(terms[np.argsort(H[i])[::-1][:10]]))

The topics that dominate are commercial and operational, not editorial:
subscription/cancel/charged/refund, pay/locked/premium, and sign-in/account/"something went
wrong". Sleep stories and courses appear as the praise topics. Nothing in the top topics is
about meditation quality. That observation is the spine of the whole analysis.

## 3. Auditable theme dictionary

Derived from the topics above. Multi-label by design — one review can express several themes.
Every rule is readable and challengeable, which is the point.

In [ ]:
THEMES = {
'paywall_gating': r"\b(locked|lock(ed|s)? behind|paywall|pay ?wall|have to pay|has to pay|need to pay|must pay|unless (you )?pay|only if you pay|everything (is )?(costs?|paid|locked)|pay for everything|paid subscription (to|required)|subscription (is )?required|not free|isn'?t free|barely (anything|any) free|almost (nothing|everything)|free (version|content|stuff) (is )?(so )?(limited|useless|minimal)|limited (free|content)|premium only|for premium|members? only)\b",
'price_value': r"(\b(expensive|overpriced|over ?priced|pricey|too much money|too costly|costs? too much|not worth|isn'?t worth|nowhere near worth|can'?t afford|unaffordable|rip ?off|ripoff|outrageous|cheaper|price is)\b|\bridiculous(ly)? (price|expensive)\b|\$\s?\d)",
'billing_cancel': r"\b(refund|charg(e|ed|es|ing)|auto.?renew|cancel(l?ed|l?ing|l?ation)?|billed|billing|unauthori[sz]ed|took my money|stole my money|money back|double charg|subscription (i|that i) (didn'?t|never)|renewed without|without (my )?(permission|consent)|scam|fraud|free trial)\b",
'login_account': r"\b(log ?in|logg(ed|ing) in|sign ?in|sign(ed|ing) in|can'?t (log|sign)|password|verification (code|email)|verify my|my account|account (is )?(locked|deleted|gone)|something went wrong|reset my|email link|otp)\b",
'technical': r"\b(crash(es|ed|ing)?|freez(e|es|ing)|frozen|bug(s|gy)?|glitch(es|y)?|won'?t (load|open|play|work)|doesn'?t (load|open|play)|not (loading|working|opening|playing)|buffer(ing)?|lag(s|gy|ging)?|error|stuck|keeps? (closing|stopping|crashing)|force clos|unusable|broken|offline|download(ing)? (issue|problem|fail))\b",
'content_repetitive': r"\b(repetitive|repeats?|repeating|same (thing|content|sessions?|stuff)|over and over|not enough (content|variety|sessions?)|need more (content|variety)|lack(s|ing)? (variety|content)|boring|bland|gets old|limited (variety|selection)|running out of)\b",
'voice_narrator': r"\b(voice|voices|accent|narrator|narration|whisper(ing|y)?|monotone|soothing voice|annoying voice|hate (the|his|her) voice|andy(?!'?s? nose)|british)\b",
'ads_upsell': r"\b(ad|ads|advert(s|ising|isement)?|pop ?up(s)?|keeps? (asking|prompting|pushing|nagging)|nag(s|ging)?|constantly (asking|pushing)|upsell|shov(e|es|ing) (it )?(down|in)|spam(s|my|ming)?|promot(e|ion|ional) (email|notification))\b",
'sleep_outcome': r"\b(sleep|asleep|insomnia|slept|bedtime|sleep stor(y|ies)|falling asleep|fall asleep|stay asleep|night ?time|dream)\b",
'anxiety_stress_outcome': r"\b(anxiety|anxious|panic|stress(ed|ful)?|overwhelm(ed|ing)?|calmer|relax(ed|ing|ation)?|breath(e|ing) exercise|depress(ed|ion)|mental health|ptsd|overthink)\b",
'efficacy_doubt': r"\b(doesn'?t (work|help|do anything)|didn'?t (work|help|do anything)|no difference|not help(ful|ing|ed)?|useless|waste of (time|money)|pointless|no effect|nothing (happened|special)|placebo|gimmick|hype)\b",
'ease_ux': r"\b(easy to (use|navigate|follow|understand)|user ?friendly|intuitive|simple (to use|interface|layout)|well designed|clean (design|interface|layout)|beautiful (design|interface)|straightforward|clunky|confusing|hard to navigate|bad (ui|ux|design)|cluttered)\b",
'support_service': r"\b(customer (service|support|care)|support team|contacted support|no (response|reply)|never (responded|replied|got back)|unresponsive|emailed (them|support)|no one (answered|responded)|help desk)\b",
'habit_streak': r"\b(streak(s)?|reminder(s)?|daily (habit|practice|routine|reminder)|notification(s)?|consistent(ly|cy)?|routine|keeps? me (on track|accountable)|motivat(e|ed|ing|ion)|gamif(y|ied|ication)|badge(s)?|progress track)\b",
'assessment_test': r"\b(personality test|assessment|questionnaire|quiz(z?es)?|my results|test results|accurate (results?|test)|survey)\b",
'course_structure': r"\b(course(s)?|program(me|mes|s)?|curriculum|structured|beginner|guided (session|meditation|journey)|learning path|step by step|teaches|instructor|lesson(s)?)\b",
}
low = df.review.str.lower()
for k, rx in THEMES.items():
    df[k] = low.str.contains(rx, regex=True)
T = list(THEMES)
print('coverage, >=1 theme:', f'{df[T].any(axis=1).mean()*100:.1f}%')


### Precision check

Dictionary methods live or die on precision, so sample and read before trusting any number.

In [ ]:
s = df[(df.app_name=='calm') & (df.rating<=2) & df.billing_cancel].sample(10, random_state=7)
for _, r in s.iterrows():
    print(f'[{r.rating}*] {r.review[:150]}')

## 4. The complaint mix

In [ ]:
f = df[df.app_name.isin(FOCUS)]
neg = (f[f.rating<=2].groupby('app_name')[T].mean()*100).round(1)
neg.loc['n_reviews'] = f[f.rating<=2].groupby('app_name').size()
neg.T.sort_values('calm', ascending=False)

In [ ]:
pos = (f[f.rating>=4].groupby('app_name')[T].mean()*100).round(1)
pos.T.sort_values('calm', ascending=False).head(8)

Two findings do the work.

**Calm's detractors are commercial, not editorial.** Billing 34.2%, paywall 26.0%, price 25.7%
of its negative reviews. Technical faults only 6.3%.

**Calm's advocates are about sleep.** 41.2% of its 4–5 star reviews mention sleep, against
19.2% for Headspace. Sleep is a genuine, defensible asset.

In [ ]:
c = df[(df.app_name=='calm') & (df.rating<=2)]
h = df[(df.app_name=='headspace') & (df.rating<=2)]
def split(x):
    comm = x.billing_cancel | x.paywall_gating | x.price_value
    prod = x.technical | x.login_account | x.content_repetitive | x.efficacy_doubt | x.voice_narrator
    return pd.Series({'commercial only': (comm&~prod).mean()*100,
                      'both': (comm&prod).mean()*100,
                      'product only': (prod&~comm).mean()*100,
                      'untagged': (~comm&~prod).mean()*100}).round(1)
pd.DataFrame({'calm': split(c), 'headspace': split(h)})

53.5% of Calm's detractors complain **only** about money and access, with no product
complaint at all. For Headspace that figure is 33.2%, and its product-only share is 26.2%.
Headspace has a product problem plus a paywall problem. Calm has essentially only a paywall problem.

## 5. The trend: grievance migrating from price to access

In [ ]:
for app in ['calm','headspace']:
    s = df[(df.app_name==app) & (df.year>=2019)]
    g = s.groupby('year').agg(n=('UID','size'), mean_rating=('rating','mean'),
                              pct_1_2=('rating', lambda x: (x<=2).mean()*100))
    t = s[s.rating<=2].groupby('year')[['billing_cancel','paywall_gating','price_value',
                                        'technical','login_account']].mean()*100
    print(f'\n{app.upper()}'); print(g.join(t.round(1)).round(1).to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
for app, col, ls in [('calm','#2a78d6','-'), ('headspace','#eb6834','--')]:
    s = df[(df.app_name==app) & (df.year>=2019)]
    y = s.groupby('year').rating.apply(lambda x: (x<=2).mean()*100)
    ax[0].plot(y.index, y.values, color=col, ls=ls, lw=2, marker='o', label=app)
ax[0].set_title('Share of reviews rated 1-2 stars'); ax[0].set_ylabel('%'); ax[0].legend()
cn = df[(df.app_name=='calm') & (df.rating<=2) & (df.year>=2019)]
for th, col, ls in [('paywall_gating','#2a78d6','-'), ('price_value','#1baf7a','--')]:
    y = cn.groupby('year')[th].mean()*100
    ax[1].plot(y.index, y.values, color=col, ls=ls, lw=2, marker='o', label=th)
ax[1].set_title("Calm detractors: access vs price"); ax[1].set_ylabel('% of negative reviews'); ax[1].legend()
for a in ax: a.grid(alpha=.25); a.set_axisbelow(True)
plt.tight_layout(); plt.show()

Between 2019 and 2022 Calm's negative share went 25.0% -> 58.1%, while the complaint
shifted: price fell from 32.5% to 21.1% of detractors, access ("everything is locked") rose
from 19.3% to 32.5%. People stopped saying it costs too much and started saying they cannot
get to anything. That is a packaging and presentation problem, not a price problem, and it is
fixable without discounting.

## 6. Why Intellect's 4.70 is not a quality signal

Intellect is sold to employers as an EAP replacement, so most users never see a paywall.
Its negative reviews contain almost no billing complaints (4.3% vs Calm's 34.2%) and are
mostly technical (15.4%). Cross-app star averages in this category are not comparable as
quality signals: the apps have different billing relationships with their users.

In [ ]:
comm = ['billing_cancel','paywall_gating','price_value']
rows = {}
for app in FOCUS:
    s = df[df.app_name==app]
    m = (s.rating<=2) & s[comm].any(axis=1)
    rows[app] = {'reviews': len(s), 'commercial 1-2 star': int(m.sum()),
                 'share of all reviews %': round(m.mean()*100,1)}
pd.DataFrame(rows).T

One in four of everything ever written about Calm on Google Play in this dataset is a
one- or two-star review about money or access. For Headspace it is one in six. For Intellect,
three in a thousand.

## 7. The say–do gap

Marketing claims collected 2026 unless dated. Review evidence runs to July 2023.

| Advertised claim | Source | What reviews show |
|---|---|---|
| "free to download and use ... free forever" | Google Play listing | 26.0% of detractors say content is locked; access complaints rose 19% -> 33% of detractors 2019-2022 |
| "Try Calm for Free" | calm.com | 13.5% of detractors mention the free trial; 15.7% mention being charged |
| "Calm your mind. Change your life." | calm.com | 34.2% of detractors are describing a billing or cancellation fight |
| "The #1 app for sleep, meditation and relaxation" | calm.com | Supported. Sleep is named in 41.2% of advocates, vs 19.2% for Headspace |
| Free tier per Calm's own help centre: 230+ meditations, 8 sleep sessions, day 1 of programs | support.calm.com | Technically generous, experienced as locked. The gap is presentation |

The free tier is not the problem. The *discovery* of the free tier is.

## 8. Recommendation

See the accompanying write-up. In short: keep the sleep claim, retire "free forever",
put the trial's terms and the free library's contents in front of the paywall rather than
behind it, and move the brand promise from a state ("calm") to an outcome the data supports
("asleep in twenty minutes").

## 9. Where Calm sits in the whole category

The per-app comparison above is only two apps. Ranking all 73 gives the finding a
denominator.

In [ ]:
COMM = ['billing_cancel','paywall_gating','price_value']
df['commercial'] = df[COMM].any(axis=1)
g = df.groupby('app_name').agg(n=('UID','size'), mean_rating=('rating','mean'))
neg = df[df.rating<=2].groupby('app_name').agg(n_neg=('UID','size'), comm_share=('commercial','mean'))
cat = g.join(neg)
cat = cat[cat.n >= 500].copy()
cat['comm_share'] = (cat.comm_share*100).round(1)
cat = cat.sort_values('comm_share', ascending=False)
print('apps with at least 500 reviews:', len(cat))
print('category median commercial share of detractors: %.1f%%' % cat.comm_share.median())
cat.head(10)

Calm ranks **first of 51** apps on the share of its detractors complaining about
money and access, at 65.0% against a category median of 18.2%. Headspace is fifth at
41.7%, Intellect thirty-sixth at 9.5%. This is not a brand with a slightly elevated
paywall complaint. It is the category's outlier.

## 10. External validation of the method

A keyword dictionary invites the obvious objection that it measures the analyst's
vocabulary rather than the users'. One way to test it is to point it at an app where
an outside authority has already established the ground truth.

Cerebral is in this dataset with 1,980 reviews. In April 2024, after the review window
closes, the FTC brought a ROSCA action against Cerebral over cancellation practices,
alleging among other things that an easy-cancel button was removed after two weeks
because cancellations spiked. If the dictionary is measuring something real, Cerebral
should stand out on cancellation without being told to.

In [ ]:
nn = df[df.rating<=2].copy()
nn['cancel'] = nn.review.str.lower().str.contains('cancel')
cc = nn.groupby('app_name').agg(n_neg=('UID','size'), cancel_pct=('cancel','mean'))
cc = cc[cc.n_neg >= 150]
cc['cancel_pct'] = (cc.cancel_pct*100).round(1)
cc.sort_values('cancel_pct', ascending=False).head(8)

Cerebral ranks first, at 32.0%, more than five times Headspace's 6.2%. The method
independently surfaces the app a regulator later sanctioned, using nothing but the word
"cancel" in user reviews. That does not prove the dictionary is right everywhere, and it
is a single case, but it is the kind of check that should be run before anyone acts on
these numbers.

## 11. How the companies respond to complaints

Google Play lets developers reply publicly. 26% of reviews in this dataset carry a
developer response, which makes response behaviour measurable. What a company chooses
to answer says what it considers a real complaint.

In [ ]:
df['has_resp'] = df.response.notna()
for app in ['calm','headspace']:
    s = df[(df.app_name==app) & (df.rating<=2)]
    print(f'\n{app.upper()}  n={len(s)}  overall response rate {s.has_resp.mean()*100:.1f}%')
    rows = {}
    for t in ['billing_cancel','paywall_gating','price_value','technical','login_account']:
        a = s[s[t]]
        rows[t] = {'n': len(a), 'response_rate_%': round(a.has_resp.mean()*100,1)}
    print(pd.DataFrame(rows).T.to_string())

Two things fall out.

Headspace answers 58.1% of its detractors; Calm answers 33.0%. Headspace's replies skew
toward things it can fix in code, 66.9% on technical faults and 68.0% on login problems.

Calm's pattern is the interesting one. It answers 39.9% of reviews complaining about
being charged, and only 26.2% of reviews complaining that content is locked, its lowest
response rate of any complaint type and well below its own 33.0% average. Calm treats a
billing dispute as a support ticket and a paywall objection as not a complaint at all.
That is a defensible reading of the numbers and worth stating as an interpretation
rather than a fact.

## 12. Where the churn goes

If users leaving Calm went to Headspace, this would be a competitive problem. They
mostly do not.

In [ ]:
subs = r'\b(you ?tube|spotify|insight ?timer)\b'
for app in ['calm','headspace']:
    s = df[(df.app_name==app) & (df.rating<=2)]
    l = s.review.str.lower()
    print(f'{app:10s} names YouTube/Spotify/Insight Timer: {l.str.contains(subs, regex=True).mean()*100:.2f}%'
          f'   (YouTube alone {l.str.contains(chr(92)+"byou ?tube"+chr(92)+"b", regex=True).mean()*100:.2f}%)')

7.3% of Calm's detractors name a free audio substitute, YouTube most often. The exit is
not to a better meditation app, it is to free audio. That matters for the recommendation:
the competitive frame is not Calm against Headspace, it is Calm against free.